In [1]:
import pandas as pd
import numpy as np
from scipy import stats

In [2]:
df = pd.read_csv('../datasets/master/master_final_2025.csv')

In [3]:
print("=== GENDER GAP BY REGION (mean gap_eys, gap_literacy, gap_edu25plus) ===")
print(df.groupby('region')[['gap_eys','gap_literacy','gap_edu25plus']].mean().round(2).sort_values('gap_literacy'))

=== GENDER GAP BY REGION (mean gap_eys, gap_literacy, gap_edu25plus) ===
              gap_eys  gap_literacy  gap_edu25plus
region                                            
Bali-Nusra       0.13         -3.66          -8.62
Jawa             0.23         -2.31          -6.68
Maluku-Papua     0.09         -2.18          -6.34
Kalimantan       0.36         -1.51          -5.54
Sumatera         0.48         -1.25          -3.04
Sulawesi         0.61         -1.22          -0.72


In [4]:
print("\n=== ANOVA: apakah gap_literacy berbeda signifikan antar region? ===")
groups = [g['gap_literacy'].values for _, g in df.groupby('region')]
f,p = stats.f_oneway(*groups)
print(f"F={f:.2f}, p={p:.4f}")


=== ANOVA: apakah gap_literacy berbeda signifikan antar region? ===
F=1.39, p=0.2557


In [5]:
groups2 = [g['gap_eys'].values for _, g in df.groupby('region')]
f2,p2 = stats.f_oneway(*groups2)
print(f"gap_eys: F={f2:.2f}, p={p2:.4f}")

gap_eys: F=4.54, p=0.0030


In [6]:
print("\n=== KORELASI: library access vs outcome ===")
sub = df.dropna(subset=['library_per_1000pupils','completion_senior_high','literacy_total'])
print("library_per_1000pupils vs completion_senior_high:", 
    stats.pearsonr(sub['library_per_1000pupils'], sub['completion_senior_high']))
print("library_per_1000pupils vs literacy_total:", 
    stats.pearsonr(df['library_per_1000pupils'], df['literacy_total']))
sub2 = df.dropna(subset=['reading_fondness_level','completion_senior_high'])
print("reading_fondness_level vs completion_senior_high:",
    stats.pearsonr(sub2['reading_fondness_level'], sub2['completion_senior_high']))
sub3 = df.dropna(subset=['reading_fondness_level','literacy_total'])
print("reading_fondness_level vs literacy_total:",
    stats.pearsonr(sub3['reading_fondness_level'], sub3['literacy_total']))


=== KORELASI: library access vs outcome ===
library_per_1000pupils vs completion_senior_high: PearsonRResult(statistic=0.17588426118250636, pvalue=0.31973804442752424)
library_per_1000pupils vs literacy_total: PearsonRResult(statistic=0.1143788778139378, pvalue=0.49411736149797203)
reading_fondness_level vs completion_senior_high: PearsonRResult(statistic=0.13035529812618157, pvalue=0.4696464543978821)
reading_fondness_level vs literacy_total: PearsonRResult(statistic=-0.03594598399798571, pvalue=0.8425788877173419)


In [7]:
print("\n=== FULL CORR MATRIX (key vars) ===")
keyvars = ['facility_ratio_jhs','facility_ratio_shs','school_density_primary',
        'library_per_1000pupils','reading_fondness_level',
        'completion_elementary','completion_junior_high','completion_senior_high',
        'literacy_total','eys_avg','gap_literacy','gap_eys']
print(df[keyvars].corr().round(2))


=== FULL CORR MATRIX (key vars) ===
                        facility_ratio_jhs  facility_ratio_shs  \
facility_ratio_jhs                    1.00                0.91   
facility_ratio_shs                    0.91                1.00   
school_density_primary               -0.30               -0.44   
library_per_1000pupils                0.22                0.06   
reading_fondness_level                0.16                0.12   
completion_elementary                 0.36                0.30   
completion_junior_high                0.36                0.38   
completion_senior_high                0.41                0.51   
literacy_total                        0.28                0.24   
eys_avg                               0.39                0.35   
gap_literacy                          0.13                0.09   
gap_eys                               0.18                0.02   

                        school_density_primary  library_per_1000pupils  \
facility_ratio_jhs            

In [9]:
# Median-split quadrant (simple, interpretable, matches user's exact question)
acc_med = df['access_index'].median()
out_med = df['outcome_index_used'].median()
def quadrant(row):
    if row['access_index']>=acc_med and row['outcome_index_used']>=out_med:
        return 'High Access - High Outcome'
    elif row['access_index']>=acc_med and row['outcome_index_used']<out_med:
        return 'High Access - Low Outcome (masalah kualitas)'
    elif row['access_index']<acc_med and row['outcome_index_used']>=out_med:
        return 'Low Access - High Outcome (efisien)'
    else:
        return 'Low Access - Low Outcome (butuh infrastruktur)'
 
df['quadrant'] = df.apply(quadrant, axis=1)
print(df['quadrant'].value_counts())
print()
for q in df['quadrant'].unique():
    print(f"--- {q} ---")
    print(df[df['quadrant']==q][['province','region','access_index','outcome_index_used']].round(2).sort_values('access_index',ascending=False).to_string(index=False))
    print()

quadrant
High Access - High Outcome                        10
Low Access - Low Outcome (butuh infrastruktur)    10
Low Access - High Outcome (efisien)                9
High Access - Low Outcome (masalah kualitas)       9
Name: count, dtype: int64

--- Low Access - High Outcome (efisien) ---
          province       region  access_index  outcome_index_used
 SULAWESI TENGGARA     Sulawesi         -0.01                0.28
         KEP. RIAU     Sumatera         -0.09                0.59
  PAPUA BARAT DAYA Maluku-Papua         -0.13                0.45
KALIMANTAN SELATAN   Kalimantan         -0.17                0.09
              ACEH     Sumatera         -0.26                0.78
    SUMATERA UTARA     Sumatera         -0.26                0.49
  KALIMANTAN TIMUR   Kalimantan         -0.37                0.63
          BENGKULU     Sumatera         -0.40                0.19
              BALI   Bali-Nusra         -0.44                0.45

--- High Access - High Outcome ---
      provin

In [11]:
from sklearn.cluster import KMeans

# KMeans clustering (data-driven, k=4) on standardized access+outcome features
feat_cols = ['facility_ratio_jhs','facility_ratio_shs','school_density_primary','z_literacy','z_eys']
X = df[feat_cols].fillna(df[feat_cols].mean())
X = (X - X.mean())/X.std()
km = KMeans(n_clusters=4, random_state=42, n_init=10)
df['kmeans_cluster'] = km.fit_predict(X)
print("\n=== KMeans cluster profile (mean values) ===")
print(df.groupby('kmeans_cluster')[['access_index','outcome_index_used']].mean().round(2))
print(df.groupby('kmeans_cluster')['province'].apply(list))


=== KMeans cluster profile (mean values) ===
                access_index  outcome_index_used
kmeans_cluster                                  
0                      -0.01                0.32
1                      -1.31               -3.51
2                       0.92                0.29
3                      -0.05               -0.05
kmeans_cluster
0    [SUMATERA UTARA, SUMATERA BARAT, RIAU, JAMBI, ...
1                     [PAPUA TENGAH, PAPUA PEGUNUNGAN]
2    [DKI JAKARTA, JAWA BARAT, BANTEN, NUSA TENGGAR...
3    [ACEH, BENGKULU, JAWA TENGAH, NUSA TENGGARA TI...
Name: province, dtype: object


In [12]:
df.to_csv('../datasets/master/master_final_2025.csv', index=False)